In [ ]:
!pip install "ray[rllib,train]" -q

# Native RL on PyTorch

## Soft Actor-Critic

In [5]:
import torch  # 引入 PyTorch 库，用于构建和训练深度学习模型
import torch.nn as nn  # PyTorch 的神经网络模块
import torch.optim as optim  # PyTorch 的优化模块，用于更新模型参数
import numpy as np  # NumPy 库，用于高效的数值计算
import gym  # OpenAI Gym 库，用于创建和交互强化学习环境
import random  # Python 的随机模块，用于随机抽样
from collections import deque  # Python 的双端队列模块，用于构建经验回放缓冲区

# 超参数设置
GAMMA = 0.99  # 折扣因子，用于计算未来奖励
TAU = 0.005  # 软更新目标网络的参数
ALPHA = 0.2  # 熵正则化系数，鼓励探索
LR = 0.001  # 学习率，用于优化器
BATCH_SIZE = 256  # 每次训练的样本数量
MEMORY_CAPACITY = 100000  # 经验回放缓冲区的最大容量

In [6]:
# 策略网络（用于生成随机的策略动作）
class PolicyNetwork(nn.Module):
    def __init__(self, state_dim, action_dim, max_action):
        super(PolicyNetwork, self).__init__()
        self.fc1 = nn.Linear(state_dim, 256)  # 第一层全连接层，输入状态维度
        self.fc2 = nn.Linear(256, 256)  # 第二层全连接层
        self.mean = nn.Linear(256, action_dim)  # 输出动作均值
        self.log_std = nn.Linear(256, action_dim)  # 输出动作的对数标准差
        self.max_action = max_action  # 动作的最大值，用于缩放

    def forward(self, state):
        x = torch.relu(self.fc1(state))  # 激活第一层
        x = torch.relu(self.fc2(x))  # 激活第二层
        mean = self.mean(x)  # 计算动作均值
        log_std = self.log_std(x).clamp(-20, 2)  # 将对数标准差限制在合理范围内
        std = torch.exp(log_std)  # 通过对数标准差计算标准差
        return mean, std  # 返回均值和标准差

    def sample(self, state):
        mean, std = self.forward(state)  # 获取动作分布的均值和标准差
        normal = torch.distributions.Normal(mean, std)  # 正态分布
        x_t = normal.rsample()  # 使用重参数化技巧采样
        y_t = torch.tanh(x_t)  # 使用 Tanh 将动作限制在 [-1, 1]
        action = y_t * self.max_action  # 缩放动作到最大值范围
        log_prob = normal.log_prob(x_t)  # 计算动作的对数概率
        log_prob -= torch.log(1 - y_t.pow(2) + 1e-6)  # Tanh 的修正项
        log_prob = log_prob.sum(dim=-1, keepdim=True)  # 对每个维度求和
        return action, log_prob  # 返回动作和对数概率

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [7]:
# Q 网络（价值函数，用于评估状态-动作对的价值）
class QNetwork(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(QNetwork, self).__init__()
        self.fc1 = nn.Linear(state_dim + action_dim, 256)  # 输入包括状态和动作
        self.fc2 = nn.Linear(256, 256)  # 第二层全连接层
        self.fc3 = nn.Linear(256, 1)  # 输出单一 Q 值

    def forward(self, state, action):
        x = torch.cat([state, action], dim=-1)  # 将状态和动作连接起来作为输入
        x = torch.relu(self.fc1(x))  # 激活第一层
        x = torch.relu(self.fc2(x))  # 激活第二层
        x = self.fc3(x)  # 输出 Q 值
        return x  # 返回 Q 值

In [8]:
# 经验回放缓冲区
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)  # 初始化一个具有固定最大容量的双端队列

    def push(self, state, action, reward, next_state, done):  # 存储经验
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):  # 随机采样
        batch = random.sample(self.buffer, batch_size)  # 随机选取 batch_size 个经验
        states, actions, rewards, next_states, dones = zip(*batch)  # 解压
        return (np.array(states), np.array(actions), np.array(rewards),
                np.array(next_states), np.array(dones))

    def __len__(self):  # 返回缓冲区当前大小
        return len(self.buffer)

In [9]:
# SAC 算法智能体
class SACAgent:
    def __init__(self, state_dim, action_dim, max_action):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # 检查是否有 GPU

        # 初始化网络
        self.actor = PolicyNetwork(state_dim, action_dim, max_action).to(self.device)  # 策略网络
        self.q1 = QNetwork(state_dim, action_dim).to(self.device)  # 第一个 Q 网络
        self.q2 = QNetwork(state_dim, action_dim).to(self.device)  # 第二个 Q 网络

        # 初始化优化器
        self.actor_optimizer = optim.Adam(self.actor.parameters(), lr=LR)  # 策略网络优化器
        self.q1_optimizer = optim.Adam(self.q1.parameters(), lr=LR)  # Q1 优化器
        self.q2_optimizer = optim.Adam(self.q2.parameters(), lr=LR)  # Q2 优化器

        # 初始化经验回放缓冲区
        self.replay_buffer = ReplayBuffer(MEMORY_CAPACITY)
        self.max_action = max_action  # 最大动作值

    def select_action(self, state):  # 根据策略选择动作
        state = torch.FloatTensor(state).to(self.device).unsqueeze(0)  # 转换状态为张量
        action, _ = self.actor.sample(state)  # 从策略中采样动作
        return action.cpu().detach().numpy()[0]  # 转换为 NumPy 格式返回

    def train(self):  # 训练智能体
        if len(self.replay_buffer) < BATCH_SIZE:  # 如果经验回放缓冲区不足，跳过训练
            return

        # 从回放缓冲区采样
        states, actions, rewards, next_states, dones = self.replay_buffer.sample(BATCH_SIZE)
        states = torch.FloatTensor(states).to(self.device)
        actions = torch.FloatTensor(actions).to(self.device)
        rewards = torch.FloatTensor(rewards).unsqueeze(1).to(self.device)
        next_states = torch.FloatTensor(next_states).to(self.device)
        dones = torch.FloatTensor(dones).unsqueeze(1).to(self.device)

        # 更新 Q 网络
        with torch.no_grad():
            next_actions, log_probs = self.actor.sample(next_states)  # 计算下一步的动作及其概率
            target_q1 = self.q1(next_states, next_actions)  # Q1 值
            target_q2 = self.q2(next_states, next_actions)  # Q2 值
            target_q = torch.min(target_q1, target_q2) - ALPHA * log_probs  # 使用最小值更新
            q_target = rewards + GAMMA * (1 - dones) * target_q  # 计算目标 Q 值

        q1_loss = ((self.q1(states, actions) - q_target) ** 2).mean()  # Q1 损失
        q2_loss = ((self.q2(states, actions) - q_target) ** 2).mean()  # Q2 损失

        self.q1_optimizer.zero_grad()  # 清空梯度
        q1_loss.backward()  # 反向传播
        self.q1_optimizer.step()  # 更新 Q1

        self.q2_optimizer.zero_grad()
        q2_loss.backward()
        self.q2_optimizer.step()

        # 更新策略网络
        new_actions, log_probs = self.actor.sample(states)
        q1_new = self.q1(states, new_actions)
        q2_new = self.q2(states, new_actions)
        q_new = torch.min(q1_new, q2_new)

        actor_loss = (ALPHA * log_probs - q_new).mean()  # 策略损失

        self.actor_optimizer.zero_grad()
        actor_loss.backward()
        self.actor_optimizer.step()

    def update_replay_buffer(self, state, action, reward, next_state, done):
        self.replay_buffer.push(state, action, reward, next_state, done)

In [10]:
# 训练循环
env = gym.make("Pendulum-v1")  # 创建环境
state_dim = env.observation_space.shape[0]  # 状态空间维度
action_dim = env.action_space.shape[0]  # 动作空间维度
max_action = float(env.action_space.high[0])  # 最大动作值

agent = SACAgent(state_dim, action_dim, max_action)  # 初始化智能体

num_episodes = 500  # 训练的总回合数
for episode in range(num_episodes):
    state = env.reset()  # 重置环境
    episode_reward = 0  # 初始化总奖励
    done = False

    while not done:
        action = agent.select_action(state)  # 根据策略选择动作
        next_state, reward, done, _ = env.step(action)  # 执行动作
        agent.update_replay_buffer(state, action, reward, next_state, done)  # 更新经验
        agent.train()  # 训练智能体
        state = next_state  # 更新状态
        episode_reward += reward  # 累积奖励

    print(f"Episode {episode}, Reward: {episode_reward}")  # 打印回合奖励

/usr/local/lib/python3.10/dist-packages/gym/core.py:317: DeprecationWarning: WARN: Initializing wrapper in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(
/usr/local/lib/python3.10/dist-packages/gym/wrappers/step_api_compatibility.py:39: DeprecationWarning: WARN: Initializing environment in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(
/usr/local/lib/python3.10/dist-packages/gym/utils/passive_env_checker.py:241: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not isinstance(terminated, (bool, np.bool8)):


Episode 0, Reward: -1762.890724103324
Episode 1, Reward: -1205.780388182603
Episode 2, Reward: -1314.8672458417443
Episode 3, Reward: -1158.9901381627567
Episode 4, Reward: -1088.1123695077433
Episode 5, Reward: -1074.779137123549
Episode 6, Reward: -1242.136118542566
Episode 7, Reward: -1557.9399558504938
Episode 8, Reward: -1009.7816393003822
Episode 9, Reward: -1374.1387393795026
Episode 10, Reward: -1749.5672551668802
Episode 11, Reward: -1344.4414055320376
Episode 12, Reward: -1120.4844425371834
Episode 13, Reward: -1444.7327286271945
Episode 14, Reward: -1567.5074015085238
Episode 15, Reward: -1612.7964709628093
Episode 16, Reward: -1559.907437599053
Episode 17, Reward: -1520.7366394158255
Episode 18, Reward: -1665.3240899600166
Episode 19, Reward: -1453.4688795793645
Episode 20, Reward: -1421.8808524279589
Episode 21, Reward: -1370.9708112496062
Episode 22, Reward: -1246.7462293089934
Episode 23, Reward: -1280.4251515494907
Episode 24, Reward: -1706.1692699622022
Episode 25, Rew

# Stable-Baselines3

In [ ]:
import gymnasium as gym

from stable_baselines3 import A2C

env = gym.make("CartPole-v1", render_mode="rgb_array")

model = A2C("MlpPolicy", env, verbose=1)
model.learn(total_timesteps=10_000)

vec_env = model.get_env()
obs = vec_env.reset()
for i in range(100):
    action, _state = model.predict(obs, deterministic=True)
    obs, reward, done, info = vec_env.step(action)
    vec_env.render("human")
    # VecEnv resets automatically
    # if done:
    #   obs = vec_env.reset()

In [ ]:
import gym
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, VecVideoRecorder
import wandb
from wandb.integration.sb3 import WandbCallback


config = {
    "policy_type": "MlpPolicy",
    "total_timesteps": 25000,
    "env_name": "CartPole-v1",
}
run = wandb.init(
    project="sb3",
    config=config,
    sync_tensorboard=True,  # auto-upload sb3's tensorboard metrics
    monitor_gym=True,  # auto-upload the videos of agents playing the game
    save_code=True,  # optional
)


def make_env():
    env = gym.make(config["env_name"])
    env = Monitor(env)  # record stats such as returns
    return env


env = DummyVecEnv([make_env])
env = VecVideoRecorder(
    env,
    f"videos/{run.id}",
    record_video_trigger=lambda x: x % 2000 == 0,
    video_length=200,
)
model = PPO(config["policy_type"], env, verbose=1, tensorboard_log=f"runs/{run.id}")
model.learn(
    total_timesteps=config["total_timesteps"],
    callback=WandbCallback(
        gradient_save_freq=100,
        model_save_path=f"models/{run.id}",
        verbose=2,
    ),
)
run.finish()

# 策略方法

### DDPG

确定性策略方法

In [6]:
import gymnasium as gym
import numpy as np

from stable_baselines3 import DDPG
from stable_baselines3.common.noise import NormalActionNoise, OrnsteinUhlenbeckActionNoise

env = gym.make("Pendulum-v1", render_mode="rgb_array")

# The noise objects for DDPG
n_actions = env.action_space.shape[-1]
action_noise = NormalActionNoise(mean=np.zeros(n_actions), sigma=0.1 * np.ones(n_actions))

model = DDPG("MlpPolicy", env, action_noise=action_noise, verbose=1)
model.learn(total_timesteps=10000, log_interval=10)
model.save("ddpg_pendulum")
vec_env = model.get_env()

del model # remove to demonstrate saving and loading

model = DDPG.load("ddpg_pendulum")

obs = vec_env.reset()
while True:
    action, _states = model.predict(obs)
    obs, rewards, dones, info = vec_env.step(action)
    env.render("human")

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
----------------------------------
| rollout/           |           |
|    ep_len_mean     | 200       |
|    ep_rew_mean     | -1.37e+03 |
| time/              |           |
|    episodes        | 10        |
|    fps             | 178       |
|    time_elapsed    | 11        |
|    total_timesteps | 2000      |
| train/             |           |
|    actor_loss      | 54.9      |
|    critic_loss     | 0.0893    |
|    learning_rate   | 0.001     |
|    n_updates       | 1800      |
----------------------------------
----------------------------------
| rollout/           |           |
|    ep_len_mean     | 200       |
|    ep_rew_mean     | -1.07e+03 |
| time/              |           |
|    episodes        | 20        |
|    fps             | 175       |
|    time_elapsed    | 22        |
|    total_timesteps | 4000      |
| train/             |           |
|    actor_loss      | 79.8   

/usr/local/lib/python3.10/dist-packages/stable_baselines3/common/save_util.py:437: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  th_object = th.load(file_content, map_locati

TypeError: Wrapper.render() takes 1 positional argument but 2 were given

### TD3

In [ ]:
import gymnasium as gym
import numpy as np

from stable_baselines3 import TD3
from stable_baselines3.common.noise import NormalActionNoise, OrnsteinUhlenbeckActionNoise

env = gym.make("Pendulum-v1", render_mode="rgb_array")

# The noise objects for TD3
n_actions = env.action_space.shape[-1]
action_noise = NormalActionNoise(mean=np.zeros(n_actions), sigma=0.1 * np.ones(n_actions))

model = TD3("MlpPolicy", env, action_noise=action_noise, verbose=1)
model.learn(total_timesteps=10000, log_interval=10)
model.save("td3_pendulum")
vec_env = model.get_env()

del model # remove to demonstrate saving and loading

model = TD3.load("td3_pendulum")

obs = vec_env.reset()
while True:
    action, _states = model.predict(obs)
    obs, rewards, dones, info = vec_env.step(action)
    vec_env.render("human")

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
----------------------------------
| rollout/           |           |
|    ep_len_mean     | 200       |
|    ep_rew_mean     | -1.43e+03 |
| time/              |           |
|    episodes        | 10        |
|    fps             | 209       |
|    time_elapsed    | 9         |
|    total_timesteps | 2000      |
| train/             |           |
|    actor_loss      | 31.3      |
|    critic_loss     | 0.083     |
|    learning_rate   | 0.001     |
|    n_updates       | 1800      |
----------------------------------
----------------------------------
| rollout/           |           |
|    ep_len_mean     | 200       |
|    ep_rew_mean     | -1.24e+03 |
| time/              |           |
|    episodes        | 20        |
|    fps             | 199       |
|    time_elapsed    | 20        |
|    total_timesteps | 4000      |
| train/             |           |
|    actor_loss      | 55.5   

## PPO

不确定性策略方法

In [ ]:
import gymnasium as gym

from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env

# Parallel environments
vec_env = make_vec_env("CartPole-v1", n_envs=4)

model = PPO("MlpPolicy", vec_env, verbose=1, )
model.learn(total_timesteps=25000)
model.save("ppo_cartpole")

del model # remove to demonstrate saving and loading

model = PPO.load("ppo_cartpole")

obs = vec_env.reset()
while True:
    action, _states = model.predict(obs)
    obs, rewards, dones, info = vec_env.step(action)
    vec_env.render("human")

/usr/local/lib/python3.10/dist-packages/tensorflow/lite/python/util.py:55: DeprecationWarning: jax.xla_computation is deprecated. Please use the AOT APIs; see https://jax.readthedocs.io/en/latest/aot.html. For example, replace xla_computation(f)(*xs) with jit(f).lower(*xs).compiler_ir('hlo'). See CHANGELOG.md for 0.4.30 for more examples.
  from jax import xla_computation as _xla_computation


Using cuda device
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 23.5     |
|    ep_rew_mean     | 23.5     |
| time/              |          |
|    fps             | 2636     |
|    iterations      | 1        |
|    time_elapsed    | 3        |
|    total_timesteps | 8192     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 30.9        |
|    ep_rew_mean          | 30.9        |
| time/                   |             |
|    fps                  | 1465        |
|    iterations           | 2           |
|    time_elapsed         | 11          |
|    total_timesteps      | 16384       |
| train/                  |             |
|    approx_kl            | 0.012675965 |
|    clip_fraction        | 0.196       |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.682      |
|    explained_variance   | -0.000163   |
|    learnin

/usr/local/lib/python3.10/dist-packages/stable_baselines3/common/save_util.py:437: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  th_object = th.load(file_content, map_locati

# 非策略方法

### DQN

In [ ]:
import gymnasium as gym

from stable_baselines3 import DQN

env = gym.make("CartPole-v1", render_mode="human")

model = DQN("MlpPolicy", env, verbose=1)
model.learn(total_timesteps=10000, log_interval=4)
model.save("dqn_cartpole")

del model # remove to demonstrate saving and loading

model = DQN.load("dqn_cartpole")

obs, info = env.reset()
while True:
    action, _states = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env.step(action)
    if terminated or truncated:
        obs, info = env.reset()

/usr/local/lib/python3.10/dist-packages/tensorflow/lite/python/util.py:55: DeprecationWarning: jax.xla_computation is deprecated. Please use the AOT APIs; see https://jax.readthedocs.io/en/latest/aot.html. For example, replace xla_computation(f)(*xs) with jit(f).lower(*xs).compiler_ir('hlo'). See CHANGELOG.md for 0.4.30 for more examples.
  from jax import xla_computation as _xla_computation


Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


/usr/local/lib/python3.10/dist-packages/pygame/pkgdata.py:25: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_stream, resource_exists
/usr/local/lib/python3.10/dist-packages/pkg_resources/__init__.py:3154: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('google')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-packages
  declare_namespace(pkg)
/usr/local/lib/python3.10/dist-packages/pkg_resources/__init__.py:3154: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('google.cloud')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-pa

----------------------------------
| rollout/            |          |
|    ep_len_mean      | 19.8     |
|    ep_rew_mean      | 19.8     |
|    exploration_rate | 0.925    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 30       |
|    time_elapsed     | 2        |
|    total_timesteps  | 79       |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 21.6     |
|    ep_rew_mean      | 21.6     |
|    exploration_rate | 0.836    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 38       |
|    time_elapsed     | 4        |
|    total_timesteps  | 173      |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 21.4     |
|    ep_rew_mean      | 21.4     |
|    exploration_rate | 0.756    |
| time/               |          |
|    episodes       

/usr/local/lib/python3.10/dist-packages/stable_baselines3/common/save_util.py:437: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  th_object = th.load(file_content, map_locati

### TD3

In [ ]:
import gymnasium as gym
import numpy as np

from stable_baselines3 import TD3
from stable_baselines3.common.noise import NormalActionNoise, OrnsteinUhlenbeckActionNoise

env = gym.make("Pendulum-v1", render_mode="rgb_array")

# The noise objects for TD3
n_actions = env.action_space.shape[-1]
action_noise = NormalActionNoise(mean=np.zeros(n_actions), sigma=0.1 * np.ones(n_actions))

model = TD3("MlpPolicy", env, action_noise=action_noise, verbose=1)
model.learn(total_timesteps=10000, log_interval=10)
model.save("td3_pendulum")
vec_env = model.get_env()

del model # remove to demonstrate saving and loading

model = TD3.load("td3_pendulum")

obs = vec_env.reset()
while True:
    action, _states = model.predict(obs)
    obs, rewards, dones, info = vec_env.step(action)
    vec_env.render("human")

## HER

In [ ]:
from stable_baselines3 import HerReplayBuffer, DDPG, DQN, SAC, TD3
from stable_baselines3.her.goal_selection_strategy import GoalSelectionStrategy
from stable_baselines3.common.envs import BitFlippingEnv

model_class = DQN  # works also with SAC, DDPG and TD3
N_BITS = 15

env = BitFlippingEnv(n_bits=N_BITS, continuous=model_class in [DDPG, SAC, TD3], max_steps=N_BITS)

# Available strategies (cf paper): future, final, episode
goal_selection_strategy = "future" # equivalent to GoalSelectionStrategy.FUTURE

# Initialize the model
model = model_class(
    "MultiInputPolicy",
    env,
    replay_buffer_class=HerReplayBuffer,
    # Parameters for HER
    replay_buffer_kwargs=dict(
        n_sampled_goal=4,
        goal_selection_strategy=goal_selection_strategy,
    ),
    verbose=1,
)

# Train the model
model.learn(1000)

model.save("./her_bit_env")
# Because it needs access to `env.compute_reward()`
# HER must be loaded with the env
model = model_class.load("./her_bit_env", env=env)

obs, info = env.reset()
for _ in range(100):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, _ = env.step(action)
    if terminated or truncated:
        obs, info = env.reset()

# RLlib

In [ ]:
from ray.rllib.algorithms.ppo import PPOConfig

config = PPOConfig()
config.environment("CartPole-v1")
config.env_runners(num_env_runners=1)
config.training(
    gamma=0.9, lr=0.01, kl_coeff=0.3, train_batch_size_per_learner=256
)

# Build a Algorithm object from the config and run 1 training iteration.
algo = config.build()
algo.train()

In [ ]:
from ray.rllib.algorithms.ppo import PPOConfig
from ray import air
from ray import tune

config = (
    PPOConfig()
    # Set the config object's env.
    .environment(env="CartPole-v1")
    # Update the config object's training parameters.
    .training(
        lr=0.001, clip_param=0.2
    )
)

tune.Tuner(
    "PPO",
    run_config=air.RunConfig(stop={"training_iteration": 1}),
    param_space=config,
).fit()